# Modeling for traffic incident hotspot analysis

This notebook applies two types of models to the Calgary traffic incident data:

1. **Spatial clustering** (DBSCAN, KMeans) to identify geographic hotspots.
2. **Classification** (Random Forest, Gradient Boosting) to predict high-incident
   periods given temporal and spatial features.

We use the `SpatialClusterAnalyzer` and `IncidentClassifier` classes from
`src/model.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score

from src.data_loader import (
    fetch_traffic_incidents, preprocess_dataframe,
    create_clustering_features, create_classification_features,
)
from src.model import (
    SpatialClusterAnalyzer, IncidentClassifier,
    save_model, save_all_artifacts,
)

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load preprocessed data

In [ ]:
import os

preprocessed_path = '../data/traffic_incidents_preprocessed.csv'
if os.path.exists(preprocessed_path):
    df = pd.read_csv(preprocessed_path, low_memory=False)
    print(f'Loaded preprocessed data: {len(df):,} rows')
else:
    raw = fetch_traffic_incidents(limit=100000)
    df = preprocess_dataframe(raw)
    print(f'Preprocessed from scratch: {len(df):,} rows')

coords = create_clustering_features(df)
print(f'Coordinate matrix: {coords.shape}')

## 2. DBSCAN with multiple eps values

`SpatialClusterAnalyzer.fit_dbscan` uses the haversine metric on radian-converted
coordinates. We sweep `eps` to see how cluster count and noise proportion change.
An eps of ~0.005 degrees (~500 m) is a reasonable starting point for urban hotspots.

In [ ]:
analyzer = SpatialClusterAnalyzer()

eps_values = [0.001, 0.003, 0.005, 0.008, 0.01]
dbscan_results = []

for eps in eps_values:
    labels = analyzer.fit_dbscan(coords, eps=eps, min_samples=10)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_pct = (labels == -1).mean() * 100
    dbscan_results.append({'eps': eps, 'n_clusters': n_clusters, 'noise_pct': round(noise_pct, 1)})
    print(f'eps={eps:.3f}  clusters={n_clusters}  noise={noise_pct:.1f}%')

dbscan_df = pd.DataFrame(dbscan_results)
dbscan_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(dbscan_df['eps'], dbscan_df['n_clusters'], 'o-', color='steelblue')
axes[0].set_xlabel('eps')
axes[0].set_ylabel('Number of clusters')
axes[0].set_title('DBSCAN: cluster count vs eps', fontsize=11)

axes[1].plot(dbscan_df['eps'], dbscan_df['noise_pct'], 's-', color='coral')
axes[1].set_xlabel('eps')
axes[1].set_ylabel('Noise (%)')
axes[1].set_title('DBSCAN: noise percentage vs eps', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Use the best eps and visualise
best_eps = 0.005
dbscan_labels = analyzer.fit_dbscan(coords, eps=best_eps, min_samples=10)

fig, ax = plt.subplots(figsize=(9, 9))
noise_mask = dbscan_labels == -1
ax.scatter(coords[noise_mask, 1], coords[noise_mask, 0], s=0.5, alpha=0.1, c='gray', label='Noise')
ax.scatter(coords[~noise_mask, 1], coords[~noise_mask, 0], s=1, alpha=0.3,
           c=dbscan_labels[~noise_mask], cmap='tab20')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'DBSCAN clusters (eps={best_eps})', fontsize=12)
ax.legend(markerscale=5)
plt.tight_layout()
plt.show()

## 3. KMeans with elbow method

We run KMeans for a range of k values and plot inertia and silhouette score
to find the optimal number of clusters.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)

k_range = range(3, 16)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(coords_scaled)
    inertias.append(km.inertia_)
    # Silhouette on a sample to keep it fast
    sample_idx = np.random.RandomState(42).choice(len(coords_scaled), size=min(5000, len(coords_scaled)), replace=False)
    sil = silhouette_score(coords_scaled[sample_idx], labels[sample_idx])
    silhouettes.append(sil)
    print(f'k={k:2d}  inertia={km.inertia_:,.0f}  silhouette={sil:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(k_range), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow method', fontsize=11)

axes[1].plot(list(k_range), silhouettes, 's-', color='teal')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette analysis', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Fit KMeans with optimal k and visualise
optimal_k = 8  # Adjust based on elbow/silhouette
kmeans_labels = analyzer.fit_kmeans(coords, n_clusters=optimal_k)
centers = analyzer.get_cluster_centers(method='kmeans')

fig, ax = plt.subplots(figsize=(9, 9))
scatter = ax.scatter(coords[:, 1], coords[:, 0], s=1, alpha=0.2,
                     c=kmeans_labels, cmap='tab10')
if centers is not None:
    ax.scatter(centers[:, 1], centers[:, 0], s=200, c='red', marker='X',
              edgecolors='black', linewidths=1.5, label='Cluster centers')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'KMeans clusters (k={optimal_k})', fontsize=12)
ax.legend(markerscale=1)
plt.tight_layout()
plt.show()

## 4. Cluster summary statistics

In [ ]:
cluster_summary = analyzer.get_cluster_summary(df.iloc[:len(kmeans_labels)], kmeans_labels)
print('KMeans cluster summary:\n')
print(cluster_summary.to_string(index=False))

## 5. Classification: predict high-incident periods

`IncidentClassifier` trains Random Forest and Gradient Boosting models to
predict the binary `high_incident` target. It performs an 80/20 stratified
split and computes accuracy, precision, recall, F1, and 5-fold CV F1.

In [ ]:
X_clf, y_clf = create_classification_features(df)

if X_clf is not None and len(X_clf) > 0:
    classifier = IncidentClassifier(random_state=42)
    clf_results = classifier.train_and_evaluate(X_clf, y_clf, test_size=0.2)

    results_table = classifier.get_results_dataframe()
    print(results_table.to_string(index=False))
else:
    print('Classification features could not be created.')

In [ ]:
# Compare models visually
if not results_table.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score']
    results_table.set_index('model')[metrics_to_plot].plot(kind='bar', ax=ax, edgecolor='black')
    ax.set_title('Classification model comparison', fontsize=12)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=15)
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

## 6. Save all model artifacts

In [ ]:
save_all_artifacts(analyzer, classifier)
print('All clustering and classification artifacts saved to ../models/')

## Summary

- DBSCAN with eps=0.005 identified dense hotspot clusters while labelling
  scattered incidents as noise. Larger eps values merge adjacent hotspots.
- KMeans elbow analysis suggests ~8 clusters provide a good balance between
  granularity and interpretability.
- Random Forest and Gradient Boosting classifiers both predict high-incident
  periods with reasonable accuracy; Gradient Boosting tends to have slightly
  higher F1 thanks to its sequential error-correction.
- All models saved for detailed evaluation in notebook 04.